# HGT Risk Assessor
**Horizontal Gene Transfer risk scoring for synthetic biology constructs**

This notebook walks you through a complete HGT risk assessment in four steps:
1. Enter your sequence
2. Configure the analysis
3. Run the pipeline
4. Read your results

No command-line knowledge required. Run each cell top-to-bottom with **Shift+Enter**.

In [ ]:
# ── Setup ──────────────────────────────────────────────────────────────────
# Run this cell once to check your environment.
import sys, subprocess, importlib

for pkg in ("ipywidgets",):
    if importlib.util.find_spec(pkg) is None:
        print(f"Installing {pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import pathlib, tempfile, json, textwrap, os

# Make sure the project root is on the path so 'src' is importable.
PROJECT_ROOT = pathlib.Path(".").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Environment ready.")

---
## Step 1 — Enter your sequence

Either **paste a sequence** directly into the text box below, or **enter a path** to an existing FASTA file on your computer.  
If you supply both, the pasted sequence takes priority.

In [ ]:
seq_area = widgets.Textarea(
    placeholder="Paste your sequence here (raw bases or FASTA format)…",
    layout=widgets.Layout(width="100%", height="160px"),
    style={"description_width": "0px"},
)
file_path_input = widgets.Text(
    placeholder="…or enter a path to a FASTA file, e.g. C:/Users/you/construct.fasta",
    layout=widgets.Layout(width="100%"),
    style={"description_width": "0px"},
)

display(
    widgets.HTML("<b>Paste sequence (raw or FASTA):</b>"),
    seq_area,
    widgets.HTML("<br><b>— or — path to a FASTA file:</b>"),
    file_path_input,
)

---
## Step 2 — Configure the analysis

Select your **host organism** and the **risk profile** that best matches your biosafety scenario.  
The donor taxon is optional but improves the taxonomic distance feature.

In [ ]:
# Common chassis organisms with their approximate GC content.
# Add more entries here if your host isn't listed.
COMMON_HOSTS: dict[str, float] = {
    "Escherichia coli K-12":         0.509,
    "Bacillus subtilis 168":          0.435,
    "Saccharomyces cerevisiae S288C": 0.381,
    "Pseudomonas putida KT2440":      0.617,
    "Caulobacter crescentus CB15":    0.671,
    "Synechocystis sp. PCC 6803":     0.476,
    "Streptomyces coelicolor A3(2)":  0.726,
    "Lactococcus lactis MG1363":      0.352,
    "Other (enter GC content below)": None,
}

PROFILE_DESCRIPTIONS = {
    "default":      "General biosafety review (balanced weights)",
    "environmental": "Open-environment / field release (transfer + persistence weighted)",
    "clinical_amr": "Clinical AMR payload (functional consequence weighted)",
}

host_dropdown = widgets.Dropdown(
    options=list(COMMON_HOSTS.keys()),
    value="Escherichia coli K-12",
    description="Host organism:",
    style={"description_width": "140px"},
    layout=widgets.Layout(width="60%"),
)
custom_gc_input = widgets.BoundedFloatText(
    value=0.50, min=0.20, max=0.85, step=0.001,
    description="Host GC content:",
    style={"description_width": "140px"},
    layout=widgets.Layout(width="35%", visibility="hidden"),
    tooltip="Only shown when 'Other' is selected above.",
)
profile_dropdown = widgets.Dropdown(
    options=[(f"{k}  —  {v}", k) for k, v in PROFILE_DESCRIPTIONS.items()],
    value="default",
    description="Risk profile:",
    style={"description_width": "140px"},
    layout=widgets.Layout(width="80%"),
)
donor_input = widgets.Text(
    placeholder="e.g. Klebsiella pneumoniae  (optional)",
    description="Donor taxon:",
    style={"description_width": "140px"},
    layout=widgets.Layout(width="60%"),
)
entrez_email_input = widgets.Text(
    placeholder="your@email.ac.uk  (needed only if host isn't in the list above)",
    description="NCBI email:",
    style={"description_width": "140px"},
    layout=widgets.Layout(width="60%"),
)

def _toggle_gc(change):
    custom_gc_input.layout.visibility = "visible" if change["new"] == "Other (enter GC content below)" else "hidden"

host_dropdown.observe(_toggle_gc, names="value")

display(
    widgets.HTML("<br>"),
    widgets.HBox([host_dropdown, custom_gc_input]),
    profile_dropdown,
    donor_input,
    entrez_email_input,
)

---
## Step 3 — Run the analysis

Click **Run Assessment** to start.  
The analysis typically completes in under a minute for sequences up to 50 kb.  
BLAST signals are skipped automatically if the databases have not been downloaded yet — the pipeline degrades gracefully.

In [ ]:
run_button = widgets.Button(
    description="  Run Assessment",
    button_style="primary",
    icon="flask",
    layout=widgets.Layout(width="220px", height="40px"),
    tooltip="Click to run the HGT risk pipeline.",
)
status_label = widgets.HTML("")
output_area  = widgets.Output()


def _prepare_fasta(seq_text: str, file_path_text: str) -> pathlib.Path | None:
    """Return a Path to a FASTA file, writing a temp file if needed."""
    seq_text = seq_text.strip()
    if seq_text:
        # Normalise: if no header line, add one.
        if not seq_text.startswith(">"):
            seq_text = ">construct\n" + seq_text
        tmp = tempfile.NamedTemporaryFile(
            suffix=".fasta", mode="w", delete=False, prefix="hgtrisk_"
        )
        tmp.write(seq_text)
        tmp.close()
        return pathlib.Path(tmp.name)
    fp = file_path_text.strip()
    if fp:
        p = pathlib.Path(fp)
        if p.exists():
            return p
    return None


def _band_colour(band: str) -> str:
    return {"low": "#2ecc71", "moderate": "#f39c12",
            "high": "#e67e22", "very_high": "#e74c3c"}.get(band, "#95a5a6")


def _render_result(result) -> str:
    """Turn a PipelineResult into a self-contained HTML summary."""
    agg  = result.aggregation
    tl   = result.three_layer

    flat_colour = {
        "Low": "#2ecc71", "Medium": "#f39c12",
        "High": "#e67e22", "Critical": "#e74c3c",
    }.get(agg.risk_level.value, "#95a5a6")

    html = textwrap.dedent(f"""
    <style>
      .hgt-box  {{ font-family: Arial, sans-serif; max-width: 820px; }}
      .hgt-band {{ display:inline-block; padding:8px 22px; border-radius:6px;
                   font-size:1.4em; font-weight:bold; color:#fff; }}
      .hgt-section {{ margin-top:18px; }}
      .hgt-section h3 {{ margin-bottom:6px; font-size:1em;
                         border-bottom:1px solid #ddd; padding-bottom:4px; }}
      table {{ border-collapse:collapse; width:100%; font-size:0.88em; }}
      th,td {{ text-align:left; padding:5px 10px; border-bottom:1px solid #eee; }}
      th {{ background:#f5f5f5; }}
      .bar-outer {{ background:#eee; border-radius:4px; height:14px; width:200px; display:inline-block; }}
      .bar-inner {{ height:14px; border-radius:4px; }}
      .pill {{ display:inline-block; padding:2px 10px; border-radius:10px;
               font-size:0.8em; color:#fff; margin:2px; }}
    </style>
    <div class='hgt-box'>
    """)

    # ── Header banner ─────────────────────────────────────────────────────
    html += f"""
    <div class='hgt-section'>
      <p style='margin:0;color:#666;font-size:0.85em;font-family:Arial'>
        Sequence: <b>{result.query.identifier}</b> &nbsp;|&nbsp;
        Length: <b>{result.query.length:,} bp</b> &nbsp;|&nbsp;
        GC: <b>{result.query.gc_content:.1%}</b> &nbsp;|&nbsp;
        Host: <b>{result.host.identifier}</b>
      </p>
    </div>
    """

    # ── Flat signal result ─────────────────────────────────────────────────
    html += f"""
    <div class='hgt-section'>
      <h3>Flat signal model</h3>
      <span class='hgt-band' style='background:{flat_colour}'>
        {agg.risk_level.value}
      </span>
      <span style='margin-left:14px;font-size:1.1em;font-family:Arial'>
        Risk index: <b>{agg.risk_index:.3f}</b>
      </span>
    """
    if agg.skipped_signals:
        html += f"""<p style='color:#e67e22;font-size:0.85em;font-family:Arial'>
          Signals skipped (databases not available or BLAST not installed):
          {', '.join(agg.skipped_signals)}.  Index may be underestimated.
        </p>"""
    html += "</div>"

    # ── Three-layer result ─────────────────────────────────────────────────
    if tl is not None:
        bc = _band_colour(tl.score_band.value)
        html += f"""
        <div class='hgt-section'>
          <h3>Three-layer HGT Risk Index &nbsp;
            <span style='font-weight:normal;font-size:0.9em;color:#888'>
              profile: {tl.weight_profile_name} &nbsp;|
              completeness: {tl.overall_completeness:.0%}
            </span>
          </h3>
          <span class='hgt-band' style='background:{bc}'>
            {tl.score_band.value.replace('_',' ').upper()}
          </span>
          <span style='margin-left:14px;font-size:1.1em;font-family:Arial'>
            HGT Risk Index: <b>{tl.hgt_risk_index:.3f}</b>
          </span>
        """

        # Layer sub-scores
        html += "<table style='margin-top:12px'>"
        html += "<tr><th>Layer</th><th>Score</th><th></th><th>Completeness</th></tr>"
        for layer in (tl.transfer_layer, tl.establishment_layer, tl.consequence_layer):
            bar_w = int(layer.layer_score * 200)
            bar_colour = _band_colour(
                "very_high" if layer.layer_score >= 0.75 else
                "high"      if layer.layer_score >= 0.50 else
                "moderate"  if layer.layer_score >= 0.25 else "low"
            )
            label = layer.layer_name.replace("_", " ").title()
            html += f"""
            <tr>
              <td>{label}</td>
              <td>{layer.layer_score:.3f}</td>
              <td>
                <div class='bar-outer'>
                  <div class='bar-inner'
                       style='width:{bar_w}px;background:{bar_colour}'></div>
                </div>
              </td>
              <td>{layer.completeness:.0%}</td>
            </tr>
            """
        html += "</table>"

        # Explanation
        html += f"""
        <div style='margin-top:14px;background:#f9f9f9;border-left:4px solid {bc};
             padding:10px 14px;border-radius:4px;font-family:Arial;font-size:0.9em'>
          {tl.explanation}
        </div>
        """

        # Top contributors / risk reducers / missing features
        if tl.top_contributors:
            html += "<div style='margin-top:10px;font-family:Arial;font-size:0.85em'><b>Key risk drivers:</b> "
            for c in tl.top_contributors:
                html += f"<span class='pill' style='background:#e74c3c'>{c.replace('_',' ')}</span>"
            html += "</div>"

        if tl.risk_reducers:
            html += "<div style='margin-top:6px;font-family:Arial;font-size:0.85em'><b>Favourable signals:</b> "
            for r in tl.risk_reducers:
                html += f"<span class='pill' style='background:#2ecc71'>{r.replace('_',' ')}</span>"
            html += "</div>"

        if tl.missing_important_features:
            html += "<div style='margin-top:6px;font-family:Arial;font-size:0.85em;color:#e67e22'>"
            html += "<b>Important features not yet available</b> (result may be underestimated): "
            html += ", ".join(f.replace("_", " ") for f in tl.missing_important_features)
            html += "</div>"

        html += "</div>"
    else:
        html += "<p style='color:#aaa;font-family:Arial;font-size:0.85em'>Three-layer model did not run.</p>"

    # ── Per-signal detail ─────────────────────────────────────────────────
    html += """
    <div class='hgt-section'>
      <h3>Signal detail</h3>
      <table>
        <tr><th>Signal</th><th>Score</th><th>Weight</th><th>Status</th></tr>
    """
    for sr in agg.signal_results:
        score_str = f"{sr.score:.3f}" if sr.score is not None else "—"
        status    = "skipped" if sr.skipped else "ok"
        status_col = "#aaa" if sr.skipped else "#2ecc71"
        html += f"""
        <tr>
          <td>{sr.signal_name.replace('_', ' ')}</td>
          <td>{score_str}</td>
          <td>{sr.weight:.2f}</td>
          <td style='color:{status_col}'>{status}</td>
        </tr>
        """
    html += "</table></div>"

    html += "</div>"  # hgt-box
    return html


def on_run(_b):
    with output_area:
        clear_output(wait=True)

        # ── Validate inputs ────────────────────────────────────────────────
        fasta_path = _prepare_fasta(seq_area.value, file_path_input.value)
        if fasta_path is None:
            display(HTML("<p style='color:red'>⚠ Please paste a sequence or enter a valid file path.</p>"))
            return

        host_name = host_dropdown.value
        host_gc   = COMMON_HOSTS.get(host_name)
        if host_gc is None:          # "Other" selected
            host_gc = custom_gc_input.value

        profile     = profile_dropdown.value
        donor_taxon = donor_input.value.strip() or None
        email       = entrez_email_input.value.strip() or ""

        display(HTML("<p style='color:#555;font-family:Arial'>Running analysis…</p>"))

        # ── Run pipeline ───────────────────────────────────────────────────
        try:
            from src.pipeline import run as pipeline_run
            from src.models import InputFormat

            result = pipeline_run(
                input_path=fasta_path,
                host_id=host_name,
                input_format=InputFormat.FASTA,
                host_gc=host_gc,
                data_dir=PROJECT_ROOT / "data",
                output_dir=PROJECT_ROOT / "results",
                no_network=(not email),
                entrez_email=email,
                weight_profile=profile,
                donor_taxon=donor_taxon,
            )
        except Exception as exc:
            display(HTML(f"<p style='color:red'><b>Pipeline error:</b> {exc}</p>"))
            raise
        finally:
            # Clean up temp file if we created one
            if seq_area.value.strip() and fasta_path.exists():
                fasta_path.unlink(missing_ok=True)

        clear_output(wait=True)
        display(HTML(_render_result(result)))

        # Show path to full HTML report
        reports = sorted(
            (PROJECT_ROOT / "results").glob("*_report.html"),
            key=lambda p: p.stat().st_mtime, reverse=True
        )
        if reports:
            display(HTML(
                f"<p style='font-family:Arial;font-size:0.85em;color:#555'>"
                f"Full report saved to: <code>{reports[0]}</code></p>"
            ))


run_button.on_click(on_run)
display(widgets.HBox([run_button, status_label]))
display(output_area)

---
## How to interpret your results

| Band | Index | Meaning |
|---|---|---|
| **Low** | < 0.25 | Minimal sequence-level HGT indicators. Standard biosafety procedures apply. |
| **Moderate** | 0.25 – 0.50 | Some indicators present. Expert review recommended before scale-up or release. |
| **High** | 0.50 – 0.75 | Multiple significant indicators. Formal contained use risk assessment required. |
| **Very High** | ≥ 0.75 | Strong HGT risk signals. Do not proceed without biosafety officer review. |

### Weight profiles
| Profile | Best for |
|---|---|
| `default` | General-purpose biosafety review |
| `environmental` | Open-environment or field release scenarios |
| `clinical_amr` | Constructs carrying antibiotic resistance payloads |

### Missing features
Several database-backed features (AMR content, virulence flags, plasmid context, transposase proximity) are not yet integrated.  
When these are unavailable, the pipeline **re-normalises** the remaining feature weights so the result isn't artificially deflated.  
The **completeness %** tells you how much of the model actually ran.  
A completeness below ~50 % means the index should be treated as a lower bound.

### BLAST databases
If BLAST+ is not installed or the databases have not been downloaded, all BLAST-derived signals are skipped automatically.  
To download the databases: open a terminal in this directory and run:
```
python data/download_databases.py --data-dir data/
```

---
*HGT Risk Assessor — University of Glasgow iGEM team*  
*This tool provides decision support and does not replace a formal risk assessment.*